# Inference speed-ups: what each technique buys, in %

Measures how much faster and smaller LLM inference gets with standard techniques, for one
request on one GPU. Where a technique fits in a few lines it is built from scratch first,
then used through its `transformers` switch. Both are measured against a baseline with
the same benchmark code.

| Section | Technique | What changes | Same output? |
|---|---|---|---|
| A1 | KV cache | Keys and values of past tokens are stored, not recomputed | Yes |
| A2 | Static cache + `torch.compile` | Fixed shapes: kernels fused, Python and launch overhead removed | Yes |
| A3 | Flash / memory-efficient attention | The N x N attention matrix is never stored | Yes |
| A4 | Speculative decoding | The big model checks several cheap guesses in one pass | Yes (greedy) |
| B1 | Weight quantization, from scratch | Fewer bits per weight; error depends on how many weights share a scale | No |
| B2 | 4-bit KV cache | Smaller cache for long contexts | No |
| B3 | 8-bit and 4-bit weights (bitsandbytes) | Smaller model in memory | No: perplexity check |

"Same output" holds up to 16-bit rounding (A1 explains why tokens can still differ late
in a sequence).

One idea behind all of them: generating a token at batch size 1 reads every weight once,
and the GPU waits for memory far more than it computes (`documents/gpu_overview.md`:
tokens/s <= memory bandwidth / model bytes). Each technique avoids work (A1), removes
overhead (A2), moves fewer bytes (A3, B2, B3) or gets several tokens out of one read of
the weights (A4).

How results are reported (summary at the end):
- **Gain %**, positive = better than the baseline of the same section.
- Speed: gain = (speed-up - 1) x 100. 2x faster = +100%.
- Memory: gain = (1 - new / baseline) x 100. Half the memory = +50%.
- Quality (perplexity): gain = (1 - new / baseline) x 100. Negative = worse predictions.
- Medians over repeated runs, with a 95% bootstrap confidence interval. With 3-5 runs the
  interval is rough, and it only covers run-to-run noise on this machine.

Plan:
  1. Setup, configuration and measurement code (PART 0-2)
  2. Part A, lower latency: A1-A4 on Qwen3-4B, Qwen3-0.6B as the draft model in A4
  3. Part B, less memory: B1-B3, perplexity on WikiText-2
  4. Summary table and chart

Assumptions:
  - One NVIDIA GPU with 16 GB or more: a Colab T4, or a rented GPU as in
    `lovecraft_lora_t4.ipynb`. Ampere or newer (L4, A100, L40S, H100) adds bf16 and the
    FlashAttention-2 kernel; on a T4 the code uses fp16 and skips what the GPU cannot run.
  - Models: `Qwen/Qwen3-4B` (about 8 GB in 16-bit) and `Qwen/Qwen3-0.6B`, same tokenizer.
    A 4B model is big enough to be limited by memory bandwidth rather than Python
    overhead; with a smaller target most gains shrink.
  - Greedy decoding, batch size 1, every run generates exactly the same number of tokens.

Limitations:
  - Batch size 1. Serving many requests (continuous batching, paged attention) is not
    covered.
  - Absolute numbers depend on GPU, clocks, drivers and library versions. Compare % within
    one run, not across machines.
  - `generate()` adds Python overhead per token, so gains are smaller than in dedicated
    engines (vLLM, TensorRT-LLM, llama.cpp).
  - Perplexity on 32k WikiText-2 tokens is a coarse quality check, not a task evaluation.
  - Run time: roughly 30-60 minutes on a T4 plus about 10 GB of model downloads
    (estimate, not measured).

## PART 0: Setup, imports and GPU check

Colab: Runtime > Change runtime type > T4 GPU (or L4 / A100). Rented VM: open this
notebook on the VM with the `LLM Sessions` kernel (see `README.md`).

Run the install cell once per session, then restart (Colab: Runtime > Restart session)
and continue from the imports cell. The restart is needed because Colab has already
loaded an older `transformers`. The cell installs the Hugging Face libraries at the
versions in `poetry.lock`, plus two quantization libraries that are not in
`pyproject.toml`:
  - `bitsandbytes`: 8-bit and 4-bit weights (B3)
  - `hqq`: 4-bit KV cache (B2)

Colab's own CUDA build of torch is kept. If transformers asks for a newer torch, install
that version too.

When done: Colab: Runtime > Disconnect and delete runtime. VM: stop it from the console.

In [ ]:
# Installs into the environment of the running kernel (Colab or VM)
%pip install -q "transformers==5.19.0" "accelerate==1.15.0" "datasets==5.1.0" "bitsandbytes==0.50.2" "hqq==0.2.8.post1"

In [ ]:
import gc
import logging
import math
import os
import sys
import time
from collections.abc import Callable
from dataclasses import asdict, dataclass
from functools import partial

import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, LogLocator, NullFormatter
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset
from IPython.display import display
from torch.nn.attention import SDPBackend, sdpa_kernel
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DynamicCache,
    GenerationConfig,
    PreTrainedConfig,
    PreTrainedModel,
    PreTrainedTokenizerBase,
    QuantizedCache,
    set_seed,
)
from transformers.generation.streamers import BaseStreamer


def make_logger(name: str) -> logging.Logger:
    """Logger that writes plain messages to the notebook output."""
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter("%(message)s"))
    logger.handlers[:] = [handler]
    logger.propagate = False
    return logger


log = make_logger("inference")

In [ ]:
assert torch.cuda.is_available(), (
    "No GPU found. Colab: Runtime > Change runtime type > T4 GPU."
)

DEVICE = torch.device("cuda", 0)
GPU_NAME = torch.cuda.get_device_name(DEVICE)
CAPABILITY = torch.cuda.get_device_capability(DEVICE)
# bf16 needs Ampere (8.0+). Do not rely on torch.cuda.is_bf16_supported(): it also
# counts software emulation and can return True on a T4
DTYPE = torch.bfloat16 if CAPABILITY >= (8, 0) else torch.float16

log.info(f"GPU:         {GPU_NAME}")
vram_gb = torch.cuda.get_device_properties(DEVICE).total_memory / 1e9
log.info(f"VRAM:        {vram_gb:.1f} GB")
log.info(f"Capability:  {CAPABILITY[0]}.{CAPABILITY[1]} (bf16, FlashAttention-2: 8.0+)")
log.info(f"Model dtype: {DTYPE}")
log.info(f"torch {torch.__version__}, CUDA {torch.version.cuda}")

## PART 1: Configuration and prompts

All settings are in one frozen dataclass. The model names can be overridden with
environment variables (`%env NAME=value` before the config cell):

| Setting | Env var | Default |
|---|---|---|
| Target model | `INFERENCE_TARGET_MODEL` | `Qwen/Qwen3-4B` |
| Draft model (A4) | `INFERENCE_DRAFT_MODEL` | `Qwen/Qwen3-0.6B` |

Draft and target must share the tokenizer. Lower `n_repeat` and `n_repeat_slow` for a
quicker but noisier run.

Two prompts:
  - **open**: a free explanation with little overlap with the prompt. The usual chat case.
  - **grounded**: add type hints to given code. The answer copies most of the input,
    which helps speculative decoding (A4).

In [ ]:
def env_str(name: str, default: str) -> str:
    """Read a string from an environment variable, else use the default."""
    return os.environ.get(name, "").strip() or default


@dataclass(frozen=True)
class BenchConfig:
    """Models, benchmark sizes and repeats for the whole notebook."""

    # Models: same tokenizer, so the small one can draft for the big one (A4)
    target_model: str = env_str("INFERENCE_TARGET_MODEL", "Qwen/Qwen3-4B")
    draft_model: str = env_str("INFERENCE_DRAFT_MODEL", "Qwen/Qwen3-0.6B")
    seed: int = 42

    # Generation: greedy, every run produces exactly n_new_tokens
    n_new_tokens: int = 256
    n_warmup: int = 1
    n_repeat: int = 5  # fast configurations
    n_repeat_slow: int = 3  # slow ones: no cache, long prompts, 8-bit weights
    n_bootstrap: int = 2000

    # A1: output lengths for the cache / no-cache loops (no cache is O(n^2): keep short)
    cache_sweep: tuple[int, ...] = (32, 64, 128, 256)
    # A3: sequence lengths for the attention kernels and for full-model prefill
    attn_seq_lens: tuple[int, ...] = (1024, 2048, 4096, 8192, 16384)
    prefill_lens: tuple[int, ...] = (1024, 2048, 4096, 8192)
    # A4: draft tokens per round (from scratch); tokens copied per round (prompt lookup)
    spec_k: int = 5
    prompt_lookup_tokens: int = 10

    # Part B: perplexity on WikiText-2 test, fixed non-overlapping windows
    ppl_window: int = 1024
    ppl_n_windows: int = 32
    # B2: long prompt for the KV cache comparison; backend "hqq" or "quanto"
    kv_prompt_len: int = 4096
    kv_bits: int = 4
    kv_backend: str = "hqq"


cfg = BenchConfig()
set_seed(cfg.seed)
RNG = np.random.default_rng(cfg.seed)  # bootstrap resampling
settings = pd.DataFrame(list(asdict(cfg).items()), columns=["setting", "value"])
display(settings.set_index("setting"))

In [ ]:
OPEN_PROMPT = (
    "Explain how a refrigerator keeps food cold. Cover the refrigerant cycle, the "
    "compressor, the condenser and the evaporator, step by step."
)

CODE_SNIPPET = '''def moving_average(values, window):
    if window <= 0:
        raise ValueError("window must be positive")
    result = []
    total = 0.0
    for i, value in enumerate(values):
        total += value
        if i >= window:
            total -= values[i - window]
        if i >= window - 1:
            result.append(total / window)
    return result


def normalize(values):
    low = min(values)
    high = max(values)
    if high == low:
        return [0.0 for _ in values]
    return [(value - low) / (high - low) for value in values]


def zscore(values):
    n = len(values)
    mean = sum(values) / n
    variance = sum((value - mean) ** 2 for value in values) / n
    std = variance ** 0.5
    return [(value - mean) / std for value in values]
'''

GROUNDED_PROMPT = (
    "Add Python type hints to every function below. Return the full code and nothing "
    "else.\n\n" + CODE_SNIPPET
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(cfg.target_model)


def chat_input_ids(tokenizer: PreTrainedTokenizerBase, prompt: str) -> torch.Tensor:
    """Render one user turn with the chat template (thinking off) and tokenize it."""
    # enable_thinking is a Qwen3 template variable; other templates ignore it
    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    ids = tokenizer(text, return_tensors="pt", add_special_tokens=False).input_ids
    return ids.to(DEVICE)


open_ids = chat_input_ids(tokenizer, OPEN_PROMPT)
grounded_ids = chat_input_ids(tokenizer, GROUNDED_PROMPT)
log.info(f"Prompt tokens: open {open_ids.shape[1]}, grounded {grounded_ids.shape[1]}")

## PART 2: Measurement code

Rules every benchmark follows:
  - **Synchronize before reading the clock.** CUDA calls return before the GPU has
    finished. Without `torch.cuda.synchronize()` you time the launch, not the work.
  - **Warm up.** The first calls pay for CUDA context setup, cuBLAS heuristics, kernel
    autotuning and compilation. They are discarded.
  - **Same work.** Greedy decoding with `min_new_tokens = max_new_tokens`: the
    end-of-text token is suppressed, so every run produces exactly `n_new_tokens`.
  - **Time to first token (TTFT) and speed from one run.** A streamer records when each
    token leaves `generate()`. TTFT = first token - start. tok/s = new tokens / total
    time, prefill included. Decode tok/s leaves out the first chunk of tokens.
  - **Peak memory** = `torch.cuda.max_memory_allocated()` after a reset: memory held by
    PyTorch tensors. `nvidia-smi` shows more (allocator cache, CUDA context).
  - **Baseline next to the technique.** Each section measures its own baseline right
    before the technique: GPU clocks drift as the card heats up (T4s throttle).

In [ ]:
GB = 1e9


def sync() -> None:
    """Wait for queued GPU work, so the clock measures the work and not the launch."""
    if DEVICE.type == "cuda":
        torch.cuda.synchronize(DEVICE)


def reset_peak() -> None:
    """Restart the peak-memory counter."""
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats(DEVICE)


def peak_bytes() -> float:
    """Peak bytes held by tensors since the last reset; NaN without a GPU."""
    if DEVICE.type != "cuda":
        return math.nan
    return float(torch.cuda.max_memory_allocated(DEVICE))


def allocated_bytes() -> float:
    """Bytes currently held by tensors; NaN without a GPU."""
    if DEVICE.type != "cuda":
        return math.nan
    return float(torch.cuda.memory_allocated(DEVICE))


def release_memory() -> None:
    """Free unreferenced tensors, return cached blocks to the driver, reset the peak."""
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    reset_peak()


def time_calls(fn: Callable[[], object], n_warmup: int, n_repeat: int) -> np.ndarray:
    """Seconds per call for n_repeat calls, after n_warmup discarded calls."""
    for _ in range(n_warmup):
        fn()
    times = np.empty(n_repeat)
    for i in range(n_repeat):
        sync()
        start = time.perf_counter()
        fn()
        sync()
        times[i] = time.perf_counter() - start
    return times


# Errors SDPA raises when the requested backend cannot run on this GPU / dtype / shape
NO_KERNEL_MESSAGES = ("no available kernel", "no viable backend")


def guarded_times(
    fn: Callable[[], object], n_warmup: int, n_repeat: int
) -> tuple[str, np.ndarray | None]:
    """time_calls(), but returns a status on out-of-memory or a missing kernel."""
    try:
        return "ok", time_calls(fn, n_warmup, n_repeat)
    except torch.OutOfMemoryError:
        status = "OOM"
    except RuntimeError as err:
        message = str(err).lower()
        if not any(text in message for text in NO_KERNEL_MESSAGES):
            raise
        status = "not supported"
    release_memory()
    return status, None

In [ ]:
def bootstrap_medians(
    x: np.ndarray, n_boot: int, rng: np.random.Generator
) -> np.ndarray:
    """Medians of n_boot resamples (with replacement) of x."""
    x = np.asarray(x, dtype=float)
    return np.median(x[rng.integers(0, len(x), size=(n_boot, len(x)))], axis=1)


def ratio_ci(
    base: np.ndarray, new: np.ndarray, n_boot: int, rng: np.random.Generator
) -> tuple[float, float]:
    """95% bootstrap CI of median(new) / median(base), independent samples."""
    ratios = bootstrap_medians(new, n_boot, rng) / bootstrap_medians(base, n_boot, rng)
    low, high = np.quantile(ratios, [0.025, 0.975])
    return float(low), float(high)


def token_agreement(a: torch.Tensor, b: torch.Tensor) -> tuple[float, int | None]:
    """Share of positions with the same token, and the first position that differs."""
    a, b = a.flatten().cpu(), b.flatten().cpu()
    n = min(len(a), len(b))
    same = a[:n] == b[:n]
    first_diff = None if bool(same.all()) else int((~same).nonzero()[0])
    return float(same.float().mean()), first_diff

In [ ]:
class TokenTimer(BaseStreamer):
    """Streamer that timestamps every chunk of tokens generate() emits."""

    def __init__(self) -> None:
        self.stamps: list[float] = []
        self.counts: list[int] = []

    def put(self, value: torch.Tensor) -> None:
        """Record arrival time and chunk size (value is already on the CPU)."""
        self.stamps.append(time.perf_counter())
        self.counts.append(value.numel())

    def end(self) -> None:
        """Nothing to flush."""


@dataclass(frozen=True)
class GenRun:
    """Timings and output of one generate() call."""

    ttft_s: float  # start of generate() -> first new token
    tok_per_s: float  # new tokens / total time, prefill included
    decode_tok_per_s: float  # tokens after the first chunk / time after the first chunk
    peak_bytes: float
    new_ids: torch.Tensor


def generate_once(
    model: PreTrainedModel, input_ids: torch.Tensor, n_new: int, **gen_kwargs: object
) -> GenRun:
    """Greedy generate() with exactly n_new new tokens; times prefill and decode."""
    timer = TokenTimer()
    reset_peak()
    sync()
    start = time.perf_counter()
    out = model.generate(
        input_ids,
        attention_mask=torch.ones_like(input_ids),
        max_new_tokens=n_new,
        min_new_tokens=n_new,  # suppresses end-of-text: every run has n_new tokens
        do_sample=False,
        streamer=timer,
        **gen_kwargs,
    )
    sync()
    total = time.perf_counter() - start
    new_ids = out[0, input_ids.shape[1]:].cpu()
    # stamps[0] is the prompt, stamps[1] the first chunk of new tokens
    first, last = timer.stamps[1], timer.stamps[-1]
    after_first = sum(timer.counts[2:])
    decode = after_first / (last - first) if after_first > 0 else math.nan
    return GenRun(first - start, len(new_ids) / total, decode, peak_bytes(), new_ids)


def bench_generate(
    model: PreTrainedModel,
    input_ids: torch.Tensor,
    n_new: int | None = None,
    n_warmup: int | None = None,
    n_repeat: int | None = None,
    **gen_kwargs: object,
) -> list[GenRun]:
    """Repeated generate_once() after warmup; unset sizes come from cfg."""
    n_new = cfg.n_new_tokens if n_new is None else n_new
    n_warmup = cfg.n_warmup if n_warmup is None else n_warmup
    n_repeat = cfg.n_repeat if n_repeat is None else n_repeat
    for _ in range(n_warmup):
        generate_once(model, input_ids, n_new, **gen_kwargs)
    return [
        generate_once(model, input_ids, n_new, **gen_kwargs) for _ in range(n_repeat)
    ]


def runs_table(named_runs: dict[str, list[GenRun]]) -> pd.DataFrame:
    """Median TTFT, speeds and peak memory per configuration, 95% CI for tok/s."""
    rows = []
    for name, runs in named_runs.items():
        tps = np.array([r.tok_per_s for r in runs])
        medians = bootstrap_medians(tps, cfg.n_bootstrap, RNG)
        low, high = np.quantile(medians, [0.025, 0.975])
        rows.append({
            "config": name,
            "TTFT ms": np.median([r.ttft_s for r in runs]) * 1e3,
            "decode tok/s": np.median([r.decode_tok_per_s for r in runs]),
            "tok/s": np.median(tps),
            "tok/s 95% CI": f"{low:.1f} - {high:.1f}",
            "peak GB": max(r.peak_bytes for r in runs) / GB,
        })
    return pd.DataFrame(rows).set_index("config").round(2)

In [ ]:
RESULTS: dict[tuple[str, str, str], dict] = {}  # (section, technique, metric) -> row


def gain_pct(ratio: float, kind: str, higher_is_better: bool) -> float:
    """Gain in % (positive = better) from ratio = new / baseline."""
    factor = ratio if higher_is_better else 1 / ratio  # > 1 means better
    return (factor - 1) * 100 if kind == "speed" else (1 - 1 / factor) * 100


def record(
    part: str,
    technique: str,
    metric: str,
    kind: str,
    base: float,
    new: float,
    higher_is_better: bool,
    ci: tuple[float, float] = (math.nan, math.nan),
) -> None:
    """Store one baseline-vs-technique comparison for the summary and log it."""
    if not (np.isfinite(base) and np.isfinite(new)) or base <= 0 or new <= 0:
        log.info(f"{part} {technique}: {metric} not measured on this device")
        return
    gain = gain_pct(new / base, kind, higher_is_better)
    gain_ci = sorted(gain_pct(r, kind, higher_is_better) for r in ci)
    RESULTS[part, technique, metric] = {
        "part": part,
        "technique": technique,
        "metric": metric,
        "kind": kind,
        "baseline": base,
        "new": new,
        "gain %": gain,
        "gain % CI low": gain_ci[0],
        "gain % CI high": gain_ci[1],
    }
    ci_text = ""
    if np.isfinite(ci[0]):
        ci_text = f" [95% CI {gain_ci[0]:+.0f}% to {gain_ci[1]:+.0f}%]"
    log.info(
        f"{part} {technique}: {metric} {base:.4g} -> {new:.4g}, "
        f"gain {gain:+.0f}%{ci_text}"
    )


def record_runs(
    part: str, technique: str, base_runs: list[GenRun], new_runs: list[GenRun]
) -> None:
    """Record the tok/s change between two sets of generate() runs, with CI."""
    base = np.array([r.tok_per_s for r in base_runs])
    new = np.array([r.tok_per_s for r in new_runs])
    ci = ratio_ci(base, new, cfg.n_bootstrap, RNG)
    record(part, technique, "tok/s", "speed", np.median(base), np.median(new), True, ci)


def record_times(
    part: str, technique: str, metric: str, base_s: np.ndarray, new_s: np.ndarray
) -> None:
    """Record a duration change (lower is better) from two timing samples."""
    ci = ratio_ci(base_s, new_s, cfg.n_bootstrap, RNG)
    base, new = np.median(base_s), np.median(new_s)
    record(part, technique, metric, "speed", base, new, False, ci)

In [ ]:
# Chart colors: categorical slots in fixed order; blue / red poles for better / worse
SERIES = ("#2a78d6", "#eb6834", "#1baf7a", "#eda100")
BETTER, WORSE = "#2a78d6", "#e34948"
INK, INK_SECONDARY, GRID, AXIS = "#0b0b0b", "#52514e", "#e1e0d9", "#c3c2b7"


def style_axes(ax: plt.Axes) -> None:
    """Recessive grid and axes: thin, light, no top or right frame."""
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color(AXIS)
    ax.tick_params(colors=INK_SECONDARY, labelsize=9)
    ax.grid(True, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.xaxis.label.set_color(INK_SECONDARY)
    ax.yaxis.label.set_color(INK_SECONDARY)
    ax.title.set_color(INK)


def plot_lines(
    ax: plt.Axes,
    df: pd.DataFrame,
    x: str,
    y: str,
    group: str,
    order: list[str],
    end_labels: bool = True,
) -> None:
    """One line per group in fixed color order; legend and optional end labels."""
    for color, name in zip(SERIES, order):
        part = df[(df[group] == name) & df[y].notna()]
        if part.empty:
            continue
        ax.plot(part[x], part[y], color=color, linewidth=2, marker="o", markersize=6,
                label=name)
        if end_labels:
            ax.annotate(name, (part[x].iloc[-1], part[y].iloc[-1]), xytext=(6, 0),
                        textcoords="offset points", va="center", fontsize=8,
                        color=INK_SECONDARY)
    ax.margins(x=0.25)
    if ax.lines:
        ax.legend(frameon=False, fontsize=8)
    style_axes(ax)

# Part A: lower latency for one request

## A0: Load the target model

Weights in 16-bit (bf16 on Ampere+, fp16 on a T4), SDPA attention (the transformers
default). Qwen3 ships sampling defaults (temperature 0.6, top-p 0.95); they are replaced by
an empty generation config, which means greedy decoding.

The spec-sheet memory bandwidth gives a speed limit for decoding: every new token reads
all weights once, so tok/s <= bandwidth / weight bytes. Real decoding stays below it
(KV cache reads, kernel launches, Python), but it shows how much room there is.

In [ ]:
# Spec-sheet memory bandwidth in GB/s, approximate (A100 40 GB; SXM and PCIe differ).
# Checked in order, so "L40S" matches before "L4" and "H100 PCIe" before "H100"
GPU_BANDWIDTH_GBS = (
    ("H200", 4800), ("H100 PCIe", 2000), ("H100", 3350), ("B200", 8000),
    ("A100", 1555), ("L40S", 864), ("RTX PRO 6000", 1792), ("A10G", 600),
    ("L4", 300), ("T4", 320),
)


def bandwidth_gbs(gpu_name: str) -> float:
    """Spec-sheet bandwidth of a known GPU; NaN if unknown."""
    for key, gbs in GPU_BANDWIDTH_GBS:
        if key in gpu_name:
            return float(gbs)
    return math.nan


def load_model(name: str, pad_token_id: int, **kwargs: object) -> PreTrainedModel:
    """Load a causal LM on DEVICE in DTYPE, eval mode, greedy generation defaults."""
    model = AutoModelForCausalLM.from_pretrained(
        name, dtype=DTYPE, attn_implementation="sdpa", device_map={"": DEVICE}, **kwargs
    )
    model.eval()
    shipped = model.generation_config
    pad = shipped.pad_token_id if shipped.pad_token_id is not None else pad_token_id
    model.generation_config = GenerationConfig(
        bos_token_id=shipped.bos_token_id,
        eos_token_id=shipped.eos_token_id,
        pad_token_id=pad,
    )
    return model

In [ ]:
target = load_model(cfg.target_model, tokenizer.pad_token_id)

conf = target.config
weights_bytes = target.get_memory_footprint()
DECODE_LIMIT = bandwidth_gbs(GPU_NAME) * 1e9 / weights_bytes  # tok/s upper bound

log.info(f"Model:           {cfg.target_model}")
log.info(f"Parameters:      {sum(p.numel() for p in target.parameters()) / 1e9:.2f} B")
log.info(f"Layers:          {conf.num_hidden_layers}")
log.info(f"Heads:           {conf.num_attention_heads} query, "
         f"{conf.num_key_value_heads} key/value, head dim {conf.head_dim}")
log.info(f"Weights:         {weights_bytes / GB:.2f} GB")
log.info(f"Decode limit:    {DECODE_LIMIT:.0f} tok/s "
         f"({bandwidth_gbs(GPU_NAME):.0f} GB/s / weight bytes)")

## A1: KV cache

To pick the next token, attention compares the newest token's query with the keys of all
earlier tokens and mixes their values. Those keys and values depend only on earlier
tokens, so they can be stored:

  - **No cache**: every step runs the model over the whole sequence again. Step t costs a
    pass over t tokens, so n new tokens cost O(n^2).
  - **KV cache**: every step runs the model over the newest token only and appends its
    keys and values. O(n) forward work, paid for in memory.

From scratch first: two greedy loops that differ only in the cache. `logits_to_keep=1`
computes the 151k-wide output projection for the last position only. Without it the
no-cache loop would also pay for logits at every earlier position, and the gain would look
larger than it is.

In [ ]:
@torch.inference_mode()
def greedy_no_cache(
    model: PreTrainedModel, input_ids: torch.Tensor, n_new: int
) -> torch.Tensor:
    """Greedy decoding that runs the model over the full sequence at every step."""
    ids = input_ids
    for _ in range(n_new):
        logits = model(ids, use_cache=False, logits_to_keep=1).logits[:, -1]
        ids = torch.cat([ids, logits.argmax(dim=-1, keepdim=True)], dim=1)
    return ids[:, input_ids.shape[1]:]


@torch.inference_mode()
def greedy_with_cache(
    model: PreTrainedModel, input_ids: torch.Tensor, n_new: int
) -> torch.Tensor:
    """Greedy decoding that feeds only the newest token and reuses cached K/V."""
    cache = DynamicCache(config=model.config)
    step_ids = input_ids  # first step: the whole prompt (prefill)
    new_ids = []
    for _ in range(n_new):
        out = model(step_ids, past_key_values=cache, use_cache=True, logits_to_keep=1)
        step_ids = out.logits[:, -1].argmax(dim=-1, keepdim=True)
        new_ids.append(step_ids)
    return torch.cat(new_ids, dim=1)

Both loops must produce the same tokens. In 16-bit, a matmul over 300 tokens and one over
1 token round differently, so after many tokens greedy decoding can flip between two
near-equal candidates. A difference late in the sequence is rounding; one in the first few
tokens would be a bug. This cell also warms up both loops.

In [ ]:
check_ids = {
    name: loop(target, open_ids, 64)
    for name, loop in (("no cache", greedy_no_cache), ("KV cache", greedy_with_cache))
}
share, first_diff = token_agreement(check_ids["no cache"], check_ids["KV cache"])
log.info(f"Same tokens with and without cache: {share:.0%} "
         f"(first difference at: {first_diff})")
log.info(tokenizer.decode(check_ids["KV cache"][0]))

Generation time against the number of new tokens: it grows quadratically without the
cache and linearly with it.

In [ ]:
LOOPS = {"no cache": greedy_no_cache, "KV cache": greedy_with_cache}

sweep_times: dict[tuple[str, int], np.ndarray] = {}
for n in cfg.cache_sweep:
    for name, loop in LOOPS.items():
        sweep_times[name, n] = time_calls(
            partial(loop, target, open_ids, n), n_warmup=0, n_repeat=cfg.n_repeat_slow
        )

sweep = pd.DataFrame([
    {"loop": name, "new tokens": n, "seconds": np.median(t),
     "ms per token": np.median(t) / n * 1e3}
    for (name, n), t in sweep_times.items()
])
display(sweep.pivot(index="new tokens", columns="loop", values="seconds").round(2))

fig, ax = plt.subplots(figsize=(7, 4), layout="constrained")
plot_lines(ax, sweep, "new tokens", "seconds", "loop", list(LOOPS))
ax.set_xlabel("new tokens")
ax.set_ylabel("seconds, median")
ax.set_title("Greedy generation time, from-scratch loops")
plt.show()

n_max = cfg.cache_sweep[-1]
record_times("A1", "KV cache, from scratch", f"time for {n_max} tokens (s)",
             sweep_times["no cache", n_max], sweep_times["KV cache", n_max])

The same switch in `generate()`: `use_cache=False` against the default dynamic cache.

In [ ]:
a1_runs = {
    "generate, use_cache=False": bench_generate(
        target, open_ids, n_repeat=cfg.n_repeat_slow, use_cache=False
    ),
    "generate, default (KV cache)": bench_generate(target, open_ids),
}
display(runs_table(a1_runs))
record_runs("A1", "KV cache, generate()", *a1_runs.values())

cached_runs = a1_runs["generate, default (KV cache)"]
cached_tps = np.median([r.decode_tok_per_s for r in cached_runs])
log.info(f"Decode with KV cache: {cached_tps:.1f} tok/s, "
         f"{cached_tps / DECODE_LIMIT:.0%} of the bandwidth limit "
         f"({DECODE_LIMIT:.0f} tok/s)")

### What the cache costs

Per token and sequence, every layer stores a key and a value vector for each KV head:

    bytes per token = 2 x layers x KV heads x head dim x bytes per value

Qwen3 uses grouped-query attention (GQA): several query heads share one KV head. With 8 KV
heads for 32 query heads the cache is 4x smaller than with one KV head per query head
(multi-head attention, MHA). At long contexts or large batches the cache, not the weights,
fills the GPU. B2 compresses it.

In [ ]:
def kv_bytes_per_token(
    config: PreTrainedConfig, bytes_per_value: float, kv_heads: int | None = None
) -> float:
    """KV cache bytes per token of one sequence: K and V for every layer and KV head."""
    head_dim = getattr(config, "head_dim", None)
    if head_dim is None:
        head_dim = config.hidden_size // config.num_attention_heads
    kv_heads = config.num_key_value_heads if kv_heads is None else kv_heads
    return 2 * config.num_hidden_layers * kv_heads * head_dim * bytes_per_value


def cache_tensor_bytes(cache: DynamicCache) -> int:
    """Bytes held by the key and value tensors of a DynamicCache."""
    return sum(
        t.numel() * t.element_size()
        for layer in cache.layers
        for t in (layer.keys, layer.values)
    )


# Measured: fill a cache with generate() and add up its tensors
cache = DynamicCache(config=target.config)
generate_once(target, open_ids, cfg.n_new_tokens, past_key_values=cache)
per_token = kv_bytes_per_token(target.config, DTYPE.itemsize)
log.info(f"Per token:  {per_token / 1e3:.0f} KB (formula)")
n_cached = cache.get_seq_length()
measured_mb = cache_tensor_bytes(cache) / 1e6
log.info(f"Cache after {n_cached} tokens: {measured_mb:.1f} MB measured, "
         f"{per_token * n_cached / 1e6:.1f} MB formula")
del cache

per_token_mha = kv_bytes_per_token(
    target.config, DTYPE.itemsize, kv_heads=target.config.num_attention_heads
)
kv_table = pd.DataFrame([
    {
        "context tokens": ctx,
        "batch": batch,
        "KV cache GB (GQA)": per_token * ctx * batch / GB,
        "KV cache GB if MHA": per_token_mha * ctx * batch / GB,
        "weights GB": weights_bytes / GB,
    }
    for ctx in (4096, 32768)
    for batch in (1, 8)
])
display(kv_table.round(1))

## A2: Static cache and `torch.compile`

At batch size 1 each decode step launches hundreds of small kernels (several per layer).
Each kernel finishes quickly, so Python dispatch and launch time are a large share of the
step. A dynamic cache grows by one position per step: tensor shapes change every step,
which blocks compiling the step into a fixed graph.

A static cache preallocates keys and values for the maximum length. Shapes stay fixed, so
`torch.compile` can fuse kernels and, in mode `"reduce-overhead"`, replay the whole step
as one CUDA graph. transformers compiles the decode step automatically when
`cache_implementation="static"`; `disable_compile=True` turns that off, which separates the
two effects.

Compilation is paid once per shape (prompt length + `max_new_tokens`) and is reported
separately. A new prompt length or output length means a new compilation, so this pays off
for servers with fixed shapes, not for one-off scripts.

In [ ]:
a2_runs = {
    "dynamic cache (baseline)": bench_generate(target, open_ids),
    "static cache, no compile": bench_generate(
        target, open_ids, cache_implementation="static", disable_compile=True
    ),
}

# The first call with compilation enabled traces and compiles the decode step
try:
    first_call = partial(
        generate_once, target, open_ids, cfg.n_new_tokens, cache_implementation="static"
    )
    compile_s = time_calls(first_call, n_warmup=0, n_repeat=1)[0]
    a2_runs["static cache + torch.compile"] = bench_generate(
        target, open_ids, n_warmup=2, cache_implementation="static"
    )
except Exception as err:  # compile support depends on GPU, driver and Triton versions
    compile_s = math.nan
    log.info(f"torch.compile failed, skipped: {type(err).__name__}: {err}")

display(runs_table(a2_runs))

In [ ]:
base_runs = a2_runs["dynamic cache (baseline)"]
record_runs("A2", "static cache", base_runs, a2_runs["static cache, no compile"])

if "static cache + torch.compile" in a2_runs:
    compiled_runs = a2_runs["static cache + torch.compile"]
    record_runs("A2", "static cache + compile", base_runs, compiled_runs)
    s_base = 1 / np.median([r.tok_per_s for r in base_runs])  # seconds per token
    s_compiled = 1 / np.median([r.tok_per_s for r in compiled_runs])
    one_off = compile_s - cfg.n_new_tokens * s_compiled
    log.info(f"First call with compilation: {compile_s:.0f} s, "
             f"of which ~{one_off:.0f} s one-off")
    if s_base > s_compiled:
        log.info(f"Compilation pays off after ~{one_off / (s_base - s_compiled):,.0f} "
                 "generated tokens with the same shapes")

# Drop compiled graphs and their memory pool before the next sections
torch.compiler.reset()
release_memory()

## A3: Flash and memory-efficient attention

Plain ("eager") attention computes S = Q K^T, an N x N matrix per head, writes it to GPU
memory, applies the softmax, writes it again, then multiplies by V. For long sequences
this matrix dominates memory (32 heads x 8192^2 x 2 bytes = 4.3 GB per layer for Qwen3-4B)
and the GPU spends its time moving it.

FlashAttention computes the same result in tiles that stay in on-chip memory (SRAM), with
a running softmax, and never writes the N x N matrix. Memory grows with N instead of N^2,
and far fewer bytes move. PyTorch's `scaled_dot_product_attention` (SDPA) has three
backends:

| Backend | How | GPUs |
|---|---|---|
| math | The plain formula, N x N matrix included | All |
| memory-efficient | Tiled, xFormers-style | All CUDA GPUs, incl. T4 |
| flash | FlashAttention-2 | Ampere+ (not T4) |

transformers uses SDPA by default; `attn_implementation="eager"` is the plain version.
Three measurements: the attention operation alone, the full model processing a long prompt
(prefill), and normal generation with a short prompt, where attention is a small share of
the work.

First a correctness check at the shortest length: every available kernel must match the
plain formula up to 16-bit rounding.

In [ ]:
def eager_attention(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    """Causal attention the textbook way: builds the full N x N score matrix."""
    n = q.shape[-2]
    scores = (q @ k.transpose(-2, -1)) / math.sqrt(q.shape[-1])
    causal = torch.ones(n, n, dtype=torch.bool, device=q.device).triu(1)
    scores = scores.masked_fill(causal, float("-inf"))
    # Softmax in fp32, as in transformers' eager attention
    return torch.softmax(scores.float(), dim=-1).to(q.dtype) @ v


def sdpa_with(
    backend: SDPBackend,
) -> Callable[[torch.Tensor, torch.Tensor, torch.Tensor], torch.Tensor]:
    """Causal scaled_dot_product_attention restricted to one backend."""

    def attention(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
        """SDPA call that may only use the chosen backend."""
        with sdpa_kernel(backend):
            return F.scaled_dot_product_attention(q, k, v, is_causal=True)

    return attention


EAGER = "eager"
ATTENTION_KERNELS = {
    EAGER: eager_attention,
    "SDPA math": sdpa_with(SDPBackend.MATH),
    "SDPA memory-efficient": sdpa_with(SDPBackend.EFFICIENT_ATTENTION),
    "SDPA flash": sdpa_with(SDPBackend.FLASH_ATTENTION),
}
FAST_KERNELS = ("SDPA memory-efficient", "SDPA flash")
HEADS, HEAD_DIM = target.config.num_attention_heads, target.config.head_dim


def random_qkv(n: int) -> tuple[torch.Tensor, ...]:
    """Reproducible random queries, keys and values of shape (1, heads, n, head dim)."""
    gen = torch.Generator(device=DEVICE).manual_seed(cfg.seed)
    shape = (1, HEADS, n, HEAD_DIM)
    return tuple(
        torch.randn(shape, device=DEVICE, dtype=DTYPE, generator=gen) for _ in range(3)
    )


q, k, v = random_qkv(cfg.attn_seq_lens[0])
reference = eager_attention(q, k, v)
for name, kernel in ATTENTION_KERNELS.items():
    status, _ = guarded_times(partial(kernel, q, k, v), n_warmup=0, n_repeat=1)
    if status == "ok":
        diff = (kernel(q, k, v) - reference).abs().max().item()
        log.info(f"{name:<22} max |difference| from eager: {diff:.1e}")
    else:
        log.info(f"{name:<22} {status} on this GPU")
del q, k, v, reference
release_memory()

The attention operation alone, one layer's worth of heads, across sequence lengths. A
kernel that runs out of memory (OOM) is not tried at longer lengths. The target model
stays loaded, so the free memory is what a real model would have left.

In [ ]:
attn_rows: list[dict] = []
attn_times: dict[tuple[str, int], np.ndarray] = {}
failed: dict[str, str] = {}  # kernel -> status at a shorter length

for n in cfg.attn_seq_lens:
    q, k, v = random_qkv(n)
    for name, kernel in ATTENTION_KERNELS.items():
        row = {"kernel": name, "seq len": n, "status": failed.get(name, "ok"),
               "ms": math.nan, "extra GB": math.nan}
        if name not in failed:
            release_memory()
            before = allocated_bytes()
            call = partial(kernel, q, k, v)
            status, times = guarded_times(call, cfg.n_warmup, cfg.n_repeat)
            row["status"] = status
            if times is None:
                failed[name] = status
            else:
                attn_times[name, n] = times
                row["ms"] = np.median(times) * 1e3
                row["extra GB"] = (peak_bytes() - before) / GB
        attn_rows.append(row)
    del q, k, v
release_memory()

attn = pd.DataFrame(attn_rows)
for value in ("ms", "extra GB", "status"):
    table = attn.pivot(index="seq len", columns="kernel", values=value)
    display(table.reindex(columns=list(ATTENTION_KERNELS)).round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
for ax, value, label in ((axes[0], "ms", "milliseconds, median"),
                         (axes[1], "extra GB", "extra peak memory, GB")):
    # Legend only: eager and SDPA math end close together, end labels would collide
    plot_lines(ax, attn, "seq len", value, "kernel", list(ATTENTION_KERNELS),
               end_labels=False)
    ax.set_xscale("log", base=2)
    ax.set_yscale("log")
    ax.set_xticks(cfg.attn_seq_lens, [str(n) for n in cfg.attn_seq_lens])
    ax.xaxis.set_minor_formatter(NullFormatter())
    ax.yaxis.set_major_locator(LogLocator(subs=(1.0, 2.0, 5.0)))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value:g}"))
    ax.yaxis.set_minor_formatter(NullFormatter())
    ax.set_xlabel("sequence length (tokens)")
    ax.set_ylabel(label)
axes[0].set_title("Attention time, one layer")
axes[1].set_title("Attention memory, one layer (lines end at OOM)")
plt.show()

# Eager vs the fastest tiled kernel, at the longest length where eager still ran
eager_ok = attn[(attn["kernel"] == EAGER) & (attn["status"] == "ok")]
if eager_ok.empty:
    log.info("Eager attention did not run at any length")
else:
    n_cmp = int(eager_ok["seq len"].max())
    fast = attn[attn["kernel"].isin(FAST_KERNELS) & (attn["seq len"] == n_cmp)
                & (attn["status"] == "ok")].sort_values("ms")
    if fast.empty:
        log.info("No tiled SDPA kernel ran on this GPU")
    else:
        best = fast.iloc[0]
        eager_row = eager_ok[eager_ok["seq len"] == n_cmp].iloc[0]
        technique = f"{best['kernel']}, attention op, {n_cmp} tokens"
        eager_ms = attn_times[EAGER, n_cmp] * 1e3
        best_ms = attn_times[best["kernel"], n_cmp] * 1e3
        record_times("A3", technique, "time (ms)", eager_ms, best_ms)
        record("A3", technique, "extra memory (GB)", "memory",
               eager_row["extra GB"], best["extra GB"], False)

Full model, prefill only: process a prompt of N tokens once (random token ids; the
content does not change the cost). `extra GB` is peak memory above the loaded weights:
activations, attention scores and the KV cache of the prompt.

In [ ]:
@torch.inference_mode()
def prefill(model: PreTrainedModel, ids: torch.Tensor) -> None:
    """Process a prompt once into a fresh KV cache; logits for the last token only."""
    model(ids, use_cache=True, logits_to_keep=1)


def random_ids(n: int) -> torch.Tensor:
    """Reproducible random token ids of length n."""
    gen = torch.Generator().manual_seed(cfg.seed)
    return torch.randint(0, target.config.vocab_size, (1, n), generator=gen).to(DEVICE)


prefill_rows: list[dict] = []
prefill_times: dict[tuple[str, int], np.ndarray] = {}
for impl in ("eager", "sdpa"):
    target.set_attn_implementation(impl)
    for n in cfg.prefill_lens:
        ids = random_ids(n)
        release_memory()
        before = allocated_bytes()
        call = partial(prefill, target, ids)
        status, times = guarded_times(call, 1, cfg.n_repeat_slow)
        row = {"attention": impl, "prompt tokens": n, "status": status,
               "prefill tok/s": math.nan, "extra GB": math.nan}
        if times is not None:
            prefill_times[impl, n] = times
            row.update({"prefill tok/s": n / np.median(times),
                        "extra GB": (peak_bytes() - before) / GB})
        prefill_rows.append(row)
        del ids
        if times is None:
            break
target.set_attn_implementation("sdpa")
release_memory()

prefill_df = pd.DataFrame(prefill_rows)
display(prefill_df.pivot(index="prompt tokens", columns="attention",
                         values=["prefill tok/s", "extra GB", "status"]).round(2))

both = [n for n in cfg.prefill_lens
        if ("eager", n) in prefill_times and ("sdpa", n) in prefill_times]
if both:
    n_cmp = both[-1]
    extra = prefill_df.set_index(["attention", "prompt tokens"])["extra GB"]
    eager_ms = prefill_times["eager", n_cmp] * 1e3
    sdpa_ms = prefill_times["sdpa", n_cmp] * 1e3
    record_times("A3", f"SDPA, prefill {n_cmp} tokens", "time (ms)", eager_ms, sdpa_ms)
    record("A3", f"SDPA, prefill {n_cmp} tokens", "extra memory (GB)", "memory",
           extra["eager", n_cmp], extra["sdpa", n_cmp], False)
is_oom = (prefill_df["attention"] == "eager") & (prefill_df["status"] == "OOM")
if is_oom.any():
    n_oom = int(prefill_df.loc[is_oom, "prompt tokens"].iloc[0])
    sdpa_status = "ok" if ("sdpa", n_oom) in prefill_times else "failed"
    log.info(f"Eager attention ran out of memory at {n_oom} prompt tokens; "
             f"SDPA there: {sdpa_status}")

Normal generation with the short open prompt. Each decode step attends over only a few
hundred positions, so attention is a small share of the step: expect little difference.

In [ ]:
a3_runs = {}
for impl in ("eager", "sdpa"):
    target.set_attn_implementation(impl)
    a3_runs[f"{impl} attention"] = bench_generate(target, open_ids)
target.set_attn_implementation("sdpa")

display(runs_table(a3_runs))
record_runs("A3", "SDPA, short-prompt generation", *a3_runs.values())

## A4: Speculative decoding

At batch size 1 a forward pass over 1 token and one over 6 tokens take about the same
time: both read all weights once, and reading dominates. Speculative decoding uses this:

  1. A small draft model proposes k tokens, one at a time (cheap).
  2. The target model scores all k proposals in one forward pass.
  3. The longest prefix that matches the target's own greedy choice is accepted, plus one
     token from the target itself (the correction, or a bonus token if all k matched).

With greedy decoding the output is the target's own output (up to rounding); only the
number of target passes drops. Expected tokens per target pass with per-token acceptance
rate a (Leviathan et al., 2023):

    E[tokens per pass] = (1 - a^(k+1)) / (1 - a)
    speed-up ~= E[tokens per pass] / (1 + k * c)

where c = draft time per token / target time per token.

**Prompt lookup decoding** needs no draft model: the proposals are copied from the prompt,
from the place where the last few generated tokens appeared before. It helps when the
answer repeats the input (editing code, extraction, quoting), and does nothing for free
writing.

In [ ]:
draft = load_model(cfg.draft_model, tokenizer.pad_token_id)
assert draft.config.vocab_size == target.config.vocab_size, (
    "Draft and target need the same vocabulary"
)
log.info(f"Draft: {cfg.draft_model}, {draft.get_memory_footprint() / GB:.2f} GB")

In [ ]:
@dataclass
class SpecStats:
    """Counts from one speculative decoding run."""

    k: int
    rounds: int = 0
    accepted: int = 0  # draft tokens accepted
    full_rounds: int = 0  # rounds in which all k proposals were accepted

    def add_round(self, n_accepted: int) -> None:
        """Count one draft-and-verify round."""
        self.rounds += 1
        self.accepted += n_accepted
        self.full_rounds += int(n_accepted == self.k)

    @property
    def acceptance_rate(self) -> float:
        """Per-token acceptance: each partial round ends with exactly one rejection."""
        rejected = self.rounds - self.full_rounds
        return self.accepted / max(self.accepted + rejected, 1)

    @property
    def tokens_per_pass(self) -> float:
        """New tokens per target pass: accepted drafts plus the target's token."""
        return (self.accepted + self.rounds) / max(self.rounds, 1)


def crop_cache(cache: DynamicCache, length: int) -> None:
    """Drop cached positions beyond length (a negative crop removes that many)."""
    excess = cache.get_seq_length() - length
    if excess > 0:
        cache.crop(-excess)


@torch.inference_mode()
def speculative_greedy(
    target: PreTrainedModel,
    draft: PreTrainedModel,
    input_ids: torch.Tensor,
    n_new: int,
    k: int,
) -> tuple[torch.Tensor, SpecStats]:
    """Greedy speculative decoding: draft proposes k tokens, target checks them."""
    target_cache = DynamicCache(config=target.config)
    draft_cache = DynamicCache(config=draft.config)
    stats = SpecStats(k=k)
    ids = input_ids
    while ids.shape[1] - input_ids.shape[1] < n_new:
        # 1. Draft k tokens; first feed whatever the draft cache has not seen yet
        draft_in = ids[:, draft_cache.get_seq_length():]
        proposals = []
        for _ in range(k):
            out = draft(draft_in, past_key_values=draft_cache, use_cache=True,
                        logits_to_keep=1)
            draft_in = out.logits[:, -1].argmax(dim=-1, keepdim=True)
            proposals.append(draft_in)
        proposals = torch.cat(proposals, dim=1)  # (1, k)

        # 2. One target pass over the unseen tokens plus all k proposals.
        #    choice[:, i] is the target's greedy token after proposal i - 1
        unseen = ids[:, target_cache.get_seq_length():]
        target_in = torch.cat([unseen, proposals], dim=1)
        logits = target(target_in, past_key_values=target_cache, use_cache=True,
                        logits_to_keep=k + 1).logits
        choice = logits.argmax(dim=-1)  # (1, k + 1)

        # 3. Accept the matching prefix, then append the target's own next token
        n_ok = int((proposals[0] == choice[0, :k]).int().cumprod(dim=0).sum())
        ids = torch.cat([ids, proposals[:, :n_ok], choice[:, n_ok:n_ok + 1]], dim=1)
        stats.add_round(n_ok)

        # 4. Forget rejected proposals; caches hold everything except the newest token
        crop_cache(target_cache, ids.shape[1] - 1)
        crop_cache(draft_cache, ids.shape[1] - 1)
    return ids[:, input_ids.shape[1]:input_ids.shape[1] + n_new], stats

From scratch: the speculative loop against the plain cached loop from A1, same target,
same tokens. The table compares the measured speed-up with the formula above, using the
measured acceptance rate and draft cost.

In [ ]:
def expected_tokens_per_pass(alpha: float, k: int) -> float:
    """Expected new tokens per target pass for per-token acceptance alpha."""
    return k + 1 if alpha >= 1 else (1 - alpha ** (k + 1)) / (1 - alpha)


PROMPTS = {"open": open_ids, "grounded": grounded_ids}
n_new, k = cfg.n_new_tokens, cfg.spec_k

# Draft cost relative to the target, per token, from the plain cached loop
t_draft = time_calls(partial(greedy_with_cache, draft, open_ids, n_new),
                     1, cfg.n_repeat_slow)
t_target = time_calls(partial(greedy_with_cache, target, open_ids, n_new),
                      0, cfg.n_repeat_slow)
cost_ratio = np.median(t_draft) / np.median(t_target)
log.info(f"Draft / target time per token: c = {cost_ratio:.2f}")

spec_rows = []
for prompt_name, ids in PROMPTS.items():
    reference = greedy_with_cache(target, ids, n_new)
    out, stats = speculative_greedy(target, draft, ids, n_new, k)  # also warms up
    share, first_diff = token_agreement(reference, out)
    t_plain = time_calls(partial(greedy_with_cache, target, ids, n_new),
                         0, cfg.n_repeat_slow)
    t_spec = time_calls(partial(speculative_greedy, target, draft, ids, n_new, k),
                        0, cfg.n_repeat_slow)
    alpha = stats.acceptance_rate
    spec_rows.append({
        "prompt": prompt_name,
        "acceptance rate": alpha,
        "tokens per target pass": stats.tokens_per_pass,
        "tokens per pass (formula)": expected_tokens_per_pass(alpha, k),
        "speed-up (formula)": expected_tokens_per_pass(alpha, k) / (1 + k * cost_ratio),
        "speed-up (measured)": np.median(t_plain) / np.median(t_spec),
        "same tokens": share,
        "first difference": first_diff,
    })
    record_times("A4", f"speculative, from scratch, {prompt_name} prompt",
                 f"time for {n_new} tokens (s)", t_plain, t_spec)
display(pd.DataFrame(spec_rows).set_index("prompt").round(2))

The same in `generate()`: `assistant_model=draft` (transformers' default number of draft
tokens per round) and `prompt_lookup_num_tokens` (no draft model), on both prompts.
transformers may print deprecation warnings from its own assisted-generation code; they do
not affect the results.

In [ ]:
SPEC_CONFIGS = {
    "target alone": {},
    "draft model": {"assistant_model": draft},
    "prompt lookup": {"prompt_lookup_num_tokens": cfg.prompt_lookup_tokens},
}

a4_runs: dict[str, list[GenRun]] = {}
for prompt_name, ids in PROMPTS.items():
    for config_name, kwargs in SPEC_CONFIGS.items():
        a4_runs[f"{prompt_name}: {config_name}"] = bench_generate(target, ids, **kwargs)


def agreement_with_target(name: str, runs: list[GenRun]) -> float:
    """Share of tokens equal to the 'target alone' run of the same prompt."""
    prompt_name = name.split(":")[0]
    reference = a4_runs[f"{prompt_name}: target alone"][0].new_ids
    return token_agreement(reference, runs[0].new_ids)[0]


table = runs_table(a4_runs)
table["same tokens as target alone"] = [
    agreement_with_target(name, runs) for name, runs in a4_runs.items()
]
display(table)

for prompt_name in PROMPTS:
    for config_name in ("draft model", "prompt lookup"):
        record_runs("A4", f"{config_name}, {prompt_name} prompt",
                    a4_runs[f"{prompt_name}: target alone"],
                    a4_runs[f"{prompt_name}: {config_name}"])

In [ ]:
del draft
release_memory()

# Part B: less memory

## B0: Data and 16-bit baseline

Quality check: perplexity on the WikiText-2 test set (Wikipedia articles), cut into 32
windows of 1024 tokens. Perplexity = exp(mean negative log-likelihood of the next token);
lower is better. Every variant sees the same windows, so the comparison is paired: the
confidence interval resamples windows.

The 16-bit target from Part A is the baseline: weights memory, peak memory during
generation, speed and perplexity.

In [ ]:
def load_eval_tokens(tokenizer: PreTrainedTokenizerBase, n_tokens: int) -> torch.Tensor:
    """First n_tokens of the WikiText-2 test set (raw text) as one token sequence."""
    ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
    # Lines keep their newlines. About 4 characters per token: 8x leaves a margin
    text = "".join(ds["text"])[: n_tokens * 8]
    ids = tokenizer(text, return_tensors="pt", add_special_tokens=False).input_ids[0]
    if len(ids) < n_tokens:
        raise ValueError(f"Got {len(ids)} WikiText-2 tokens, need {n_tokens}")
    return ids[:n_tokens]


@torch.inference_mode()
def window_nll(model: PreTrainedModel, windows: torch.Tensor) -> np.ndarray:
    """Mean next-token negative log-likelihood per window, windows independent."""
    nll = []
    for window in windows:
        ids = window[None].to(model.device)
        logits = model(ids).logits[0, :-1].float()
        nll.append(F.cross_entropy(logits, ids[0, 1:]).item())
    return np.array(nll)


def paired_ppl_ratio_ci(
    base_nll: np.ndarray, new_nll: np.ndarray, n_boot: int, rng: np.random.Generator
) -> tuple[float, float]:
    """95% bootstrap CI of the perplexity ratio; both resample the same windows."""
    idx = rng.integers(0, len(base_nll), size=(n_boot, len(base_nll)))
    ratios = np.exp(new_nll[idx].mean(axis=1) - base_nll[idx].mean(axis=1))
    low, high = np.quantile(ratios, [0.025, 0.975])
    return float(low), float(high)


@dataclass(frozen=True)
class VariantResult:
    """Memory, quality and speed of one model variant."""

    weights_bytes: float
    nll: np.ndarray
    runs: list[GenRun]

    @property
    def perplexity(self) -> float:
        """exp(mean NLL) over all windows."""
        return float(np.exp(self.nll.mean()))

    @property
    def peak_bytes(self) -> float:
        """Highest peak memory over the generation runs."""
        return max(r.peak_bytes for r in self.runs)


def measure_variant(model: PreTrainedModel) -> VariantResult:
    """Perplexity, generation speed and memory of a loaded model."""
    release_memory()
    nll = window_nll(model, ppl_windows)
    assert np.isfinite(nll).all(), (
        "Non-finite loss: fp16 overflow. Use a GPU with bf16 (L4, A100)."
    )
    runs = bench_generate(model, open_ids, n_repeat=cfg.n_repeat_slow)
    return VariantResult(float(model.get_memory_footprint()), nll, runs)

In [ ]:
n_ppl = cfg.ppl_window * cfg.ppl_n_windows
eval_ids = load_eval_tokens(tokenizer, max(n_ppl, cfg.kv_prompt_len))
ppl_windows = eval_ids[:n_ppl].view(cfg.ppl_n_windows, cfg.ppl_window)

baseline = measure_variant(target)
log.info(f"16-bit: weights {baseline.weights_bytes / GB:.2f} GB, "
         f"peak {baseline.peak_bytes / GB:.2f} GB, "
         f"perplexity {baseline.perplexity:.2f}")

## B1: Weight quantization from scratch

Store each weight as a small integer and a shared scale: w ~= scale x q. Symmetric absmax:

    scale = max |w| / q_max,   q = round(w / scale),   q_max = 127 (8-bit) or 7 (4-bit)

What matters is how many weights share one scale. A single large weight (an outlier) sets
the scale for its whole block; every other weight in the block then lands on a few levels,
many on zero. Smaller blocks contain the damage at a cost: a 16-bit scale per 64 weights
adds 0.25 bits per weight.

Measured on one real weight matrix (the MLP down-projection of the middle layer) and on
its real input activations from the open prompt. "Output error" is the relative error of
the layer's output, which is what the next layer sees.

In [ ]:
def quantize_absmax(
    w: torch.Tensor, bits: int, block: int
) -> tuple[torch.Tensor, torch.Tensor]:
    """Symmetric absmax quantization with one scale per block of consecutive weights."""
    q_max = 2 ** (bits - 1) - 1
    blocks = w.float().reshape(-1, block)
    scale = blocks.abs().amax(dim=1, keepdim=True).clamp(min=1e-12) / q_max
    q = torch.round(blocks / scale).clamp(-q_max, q_max).to(torch.int8)
    return q, scale


def dequantize(q: torch.Tensor, scale: torch.Tensor, shape: torch.Size) -> torch.Tensor:
    """Integers times their block scale, back in the original shape (fp32)."""
    return (q.float() * scale).reshape(shape)


def relative_error(approx: torch.Tensor, exact: torch.Tensor) -> float:
    """Frobenius norm of the error relative to the norm of the exact tensor."""
    return float((approx - exact).norm() / exact.norm())


def granularity_table(w: torch.Tensor, x: torch.Tensor) -> pd.DataFrame:
    """Size and error of 8-bit and 4-bit absmax quantization for several block sizes."""
    blocks = {"whole tensor": w.numel(), "per row": w.shape[1],
              "per 128 weights": 128, "per 64 weights": 64}
    exact_out = x @ w.float().T
    rows = []
    for bits in (8, 4):
        for name, block in blocks.items():
            q, scale = quantize_absmax(w, bits, block)
            w_hat = dequantize(q, scale, w.shape)
            bits_per_weight = bits + 16 * scale.numel() / w.numel()
            rows.append({
                "bits": bits,
                "one scale per": name,
                "bits per weight": bits_per_weight,
                "memory saved vs 16-bit %": (1 - bits_per_weight / 16) * 100,
                "weights rounded to 0 %": float((q == 0).float().mean()) * 100,
                "weight error %": relative_error(w_hat, w.float()) * 100,
                "output error %": relative_error(x @ w_hat.T, exact_out) * 100,
            })
    return pd.DataFrame(rows).set_index(["bits", "one scale per"])

In [ ]:
layer_idx = target.config.num_hidden_layers // 2
down_proj = target.model.layers[layer_idx].mlp.down_proj
captured: dict[str, torch.Tensor] = {}


def save_input(
    _module: torch.nn.Module, inputs: tuple[torch.Tensor, ...], _output: torch.Tensor
) -> None:
    """Forward hook: keep the input activations of the inspected layer."""
    captured["x"] = inputs[0].detach()


handle = down_proj.register_forward_hook(save_input)
with torch.inference_mode():
    target(open_ids, logits_to_keep=1)
handle.remove()

w = down_proj.weight.detach()
x = captured.pop("x")[0].float()  # (prompt tokens, in features)
abs_w = w.float().abs().flatten().cpu().numpy()
log.info(f"Layer {layer_idx} mlp.down_proj: weight {tuple(w.shape)}, "
         f"input {tuple(x.shape)}")
log.info(f"Largest |w| = {abs_w.max():.3f}; "
         f"99.9% of |w| are below {np.quantile(abs_w, 0.999):.3f}")
display(granularity_table(w, x).round(2))
del w, x, abs_w, down_proj
release_memory()

Expect 8-bit to be nearly harmless at every block size, and 4-bit with one scale for the
whole tensor to round most weights to zero. Production 4-bit formats (NF4 in B3, GPTQ, AWQ)
use blocks of 64-128 weights.

## B2: 4-bit KV cache

The same idea applied to the cache. transformers' `QuantizedCache` (after the KIVI paper)
keeps the newest 128 tokens in 16-bit and stores older keys and values in 4-bit, with a
scale and zero-point per group of 64 values. Every attention step dequantizes them again:
memory goes down, speed goes down too.

Backend `"hqq"` (default) runs in plain PyTorch. The alternative `"quanto"` needs
`%pip install optimum-quanto==0.2.7` and compiles a CUDA extension on first use (needs
`nvcc`, takes minutes).

Prompt: 4096 tokens of WikiText-2 text, then 256 generated tokens. Measured: memory held by
the cache after generation, peak memory, speed, and whether the greedy output stays the
same. The cache is small next to 8 GB of weights at this length; it dominates at long
contexts and large batches (see the table in A1).

In [ ]:
long_ids = eval_ids[: cfg.kv_prompt_len][None].to(DEVICE)
KV_KINDS = ("16-bit", f"{cfg.kv_bits}-bit")


def make_cache(kind: str) -> DynamicCache | QuantizedCache:
    """Empty 16-bit or quantized KV cache for the target model."""
    if kind == "16-bit":
        return DynamicCache(config=target.config)
    return QuantizedCache(
        backend=cfg.kv_backend, config=target.config, nbits=cfg.kv_bits
    )


def run_with_cache(kind: str) -> tuple[GenRun, float]:
    """Generate from the long prompt into a fresh cache; return run and cache bytes."""
    cache = make_cache(kind)
    release_memory()
    before = allocated_bytes()
    run = generate_once(target, long_ids, cfg.n_new_tokens, past_key_values=cache)
    return run, allocated_bytes() - before  # the cache is still alive here


kv_runs: dict[str, list[GenRun]] = {}
kv_held: dict[str, float] = {}
for kind in KV_KINDS:
    try:
        run_with_cache(kind)  # warmup
        measured = [run_with_cache(kind) for _ in range(cfg.n_repeat_slow)]
    except ImportError as err:
        log.info(f"{kind} cache skipped: {err}")
        continue
    kv_runs[kind] = [run for run, _ in measured]
    kv_held[kind] = float(np.median([held for _, held in measured]))
release_memory()

if kv_runs:
    table = runs_table(kv_runs)
    table["KV cache GB"] = [kv_held[kind] / GB for kind in kv_runs]
    display(table.round(3))
n_cached = cfg.kv_prompt_len + cfg.n_new_tokens
formula_gb = kv_bytes_per_token(target.config, DTYPE.itemsize) * n_cached / GB
log.info(f"16-bit cache by formula: {formula_gb:.3f} GB")

if len(kv_runs) == 2:
    base_kind, quant_kind = KV_KINDS
    technique = f"{cfg.kv_bits}-bit KV cache, {cfg.kv_prompt_len}-token prompt"
    record("B2", technique, "KV cache memory (GB)", "memory",
           kv_held[base_kind] / GB, kv_held[quant_kind] / GB, False)
    record("B2", technique, "peak memory (GB)", "memory",
           max(r.peak_bytes for r in kv_runs[base_kind]) / GB,
           max(r.peak_bytes for r in kv_runs[quant_kind]) / GB, False)
    record_runs("B2", technique, kv_runs[base_kind], kv_runs[quant_kind])
    share, first_diff = token_agreement(
        kv_runs[base_kind][0].new_ids, kv_runs[quant_kind][0].new_ids
    )
    log.info(f"Same tokens as with the 16-bit cache: {share:.0%} "
             f"(first difference at: {first_diff})")

## B3: 8-bit and 4-bit weights with bitsandbytes

Production kernels quantize the whole model at load time:

| Format | How | Expect |
|---|---|---|
| int8 (LLM.int8, `load_in_8bit`) | int8 weights with one scale per row. Input columns with outliers above `llm_int8_threshold` (6.0) are multiplied in 16-bit separately | Quality close to 16-bit. Usually slower: every matmul is split in two and inputs are quantized on the fly |
| NF4 (`load_in_4bit`, `nf4`) | 4-bit codes from a table shaped like a normal distribution, one scale per 64 weights, scales quantized again (double quantization). Dequantized to 16-bit right before each matmul | Smallest. Speed depends on the kernel: 4x fewer bytes to read, but dequantization adds work |

Embeddings, norms and the output layer stay in 16-bit, so the total drops by less than
50% / 75%. The 16-bit target is freed first; each variant is loaded, measured and freed in
turn.

In [ ]:
del target
release_memory()

QUANT_CONFIGS = {
    "int8 (LLM.int8)": BitsAndBytesConfig(load_in_8bit=True),
    "nf4": BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=DTYPE,
    ),
}

variants: dict[str, VariantResult] = {"16-bit": baseline}
for name, quant_config in QUANT_CONFIGS.items():
    model = load_model(
        cfg.target_model, tokenizer.pad_token_id, quantization_config=quant_config
    )
    variants[name] = measure_variant(model)
    del model
    release_memory()

In [ ]:
quant_table = pd.DataFrame([
    {
        "weights": name,
        "weights GB": v.weights_bytes / GB,
        "peak GB": v.peak_bytes / GB,
        "tok/s": np.median([r.tok_per_s for r in v.runs]),
        "perplexity": v.perplexity,
        "same tokens as 16-bit": token_agreement(
            baseline.runs[0].new_ids, v.runs[0].new_ids
        )[0],
    }
    for name, v in variants.items()
]).set_index("weights")
display(quant_table.round(2))

for name, v in variants.items():
    if name == "16-bit":
        continue
    technique = f"{name} weights"
    record("B3", technique, "weights memory (GB)", "memory",
           baseline.weights_bytes / GB, v.weights_bytes / GB, False)
    record("B3", technique, "peak memory (GB)", "memory",
           baseline.peak_bytes / GB, v.peak_bytes / GB, False)
    record_runs("B3", technique, baseline.runs, v.runs)
    ppl_ci = paired_ppl_ratio_ci(baseline.nll, v.nll, cfg.n_bootstrap, RNG)
    record("B3", technique, "perplexity", "quality",
           baseline.perplexity, v.perplexity, False, ppl_ci)

# Summary

Every comparison recorded above, against the baseline of its own section. Gain %:
positive = better. Speed: 2x faster = +100%. Memory: half = +50%. Perplexity: negative =
worse predictions. Whiskers show the 95% bootstrap interval where repeated runs exist.

In [ ]:
summary = pd.DataFrame(RESULTS.values())
summary_table = summary.set_index(["part", "technique", "metric"]).drop(columns="kind")
display(summary_table.round(3))


def plot_gains(summary: pd.DataFrame) -> None:
    """Horizontal bars of gain % per comparison, one panel per kind of metric."""
    titles = {"speed": "Speed: % faster (2x = +100%)",
              "memory": "Memory: % less (half = +50%)",
              "quality": "Quality: % lower perplexity (negative = worse)"}
    kinds = [kind for kind in titles if (summary["kind"] == kind).any()]
    sizes = [int((summary["kind"] == kind).sum()) for kind in kinds]
    height = 0.35 * sum(sizes) + 1.3 * len(kinds)
    _, axes = plt.subplots(len(kinds), 1, figsize=(10, height),
                           gridspec_kw={"height_ratios": sizes}, squeeze=False,
                           layout="constrained")
    for ax, kind in zip(axes[:, 0], kinds):
        rows = summary[summary["kind"] == kind].iloc[::-1]  # first recorded on top
        y = np.arange(len(rows))
        gain = rows["gain %"].to_numpy()
        low, high = rows["gain % CI low"].to_numpy(), rows["gain % CI high"].to_numpy()
        ax.barh(y, gain, height=0.6, color=np.where(gain >= 0, BETTER, WORSE))
        has_ci = np.isfinite(low)
        ax.errorbar(gain[has_ci], y[has_ci],
                    xerr=[gain[has_ci] - low[has_ci], high[has_ci] - gain[has_ci]],
                    fmt="none", ecolor=INK_SECONDARY, elinewidth=1, capsize=2)
        for yi, g, lo, hi in zip(y, gain, low, high):
            end = (hi if g >= 0 else lo) if np.isfinite(lo) else g
            ax.annotate(f"{g:+.0f}%", (end, yi), xytext=(4 if g >= 0 else -4, 0),
                        textcoords="offset points", ha="left" if g >= 0 else "right",
                        va="center", fontsize=8, color=INK)
        labels = rows["part"] + "  " + rows["technique"] + ": " + rows["metric"]
        ax.set_yticks(y, labels, fontsize=8)
        ax.axvline(0, color=AXIS, linewidth=1)
        ax.margins(x=0.15)
        ax.set_title(titles[kind], loc="left", fontsize=10)
        style_axes(ax)
        ax.grid(axis="y", visible=False)
    axes[-1, 0].set_xlabel("gain %, positive = better than the section's baseline")
    plt.show()


plot_gains(summary)

What to look for. Expected pattern from the literature, not a result; your numbers are in
the table above:
  - **A1**: the largest speed gain, and it grows with the output length (O(n^2) vs O(n)).
  - **A2**: compilation removes overhead, so it helps most where overhead is a large share
    of a step (small models, fast GPUs). The one-off compile time only pays off for fixed
    shapes and many tokens.
  - **A3**: large memory and speed gains for long prompts, eager may run out of memory;
    close to 0% for short-prompt generation.
  - **A4**: higher gains on the grounded prompt; prompt lookup close to 0% on the open
    prompt. The output stays the target's own.
  - **B2**: the cache shrinks by about 3-4x; peak memory barely moves at 4k tokens next to
    8 GB of weights; speed drops.
  - **B3**: weights about -45% (int8) and -65% (NF4); perplexity slightly worse; int8 usually
    slower, NF4 speed depends on the GPU and kernel.

Speed and memory are bought separately: A-techniques cost memory (A1) or nothing, B-techniques
cost speed or quality. Choose by the bottleneck: latency for one user, memory for long
contexts and many users.

In [ ]:
release_memory()
log.info("Done. Colab: Runtime > Disconnect and delete runtime. "
         "VM: stop it from the console.")